# Demo 04 : History and time travel

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · state-history

We build a game and read its checkpoint history. The Producer picks `golden` on purpose : a bad pick, because the brief asks for a second chance and golden stars only give a prize. The Coder is a live model call and the Tester runs real tests.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]
import importlib.util
from pathlib import Path

kit = Path(data_folder("game-studio/kit"))
spec = importlib.util.spec_from_file_location("harness", kit / "harness.py")
harness = importlib.util.module_from_spec(spec)
spec.loader.exec_module(harness)

TEMPLATE = (kit / "template.html").read_text(encoding="utf-8")
RULES = (kit / "rules.md").read_text(encoding="utf-8")
TWISTS = json.loads((kit / "twists.json").read_text(encoding="utf-8"))
BRIEFS = json.loads((kit / "briefs.json").read_text(encoding="utf-8"))

In [3]:
from typing import TypedDict
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

brief = BRIEFS[0]["brief"]
coder_llm = make_llm(max_tokens=8000)

def twist_text(twist):
    return f"{TWISTS[twist]['name']}\n" + "\n".join(f"- {rule}" for rule in TWISTS[twist]["rules"])

def fence(text):
    blocks = re.findall(r"```(?:javascript|js)?\s*\n(.*?)```", text, re.S)
    return (max(blocks, key=len) if blocks else text).replace("/* LOGIC:START */", "").replace("/* LOGIC:END */", "").strip()

class Board(TypedDict):
    brief: str
    twist: str
    logic: str
    report: str

calls = []
def producer(board: Board):
    return {"twist": "golden"}

def coder(board: Board):
    calls.append(board["twist"])
    ask = (f"{RULES}\n\n## The twist : {twist_text(board['twist'])}\n\n"
           f"## The logic block\n\n```javascript\n{harness.logic_of(TEMPLATE)}\n```\n\n"
           "Return the complete logic block with every rule and the twist in it, in one javascript code block. "
           "Keep every helper. Keep the function names init and step.")
    reply = coder_llm.invoke([SystemMessage("You are the Coder at Comet Arcade. You write the pure game logic of a one-file canvas game."),
                              HumanMessage(ask)])
    return {"logic": fence(reply.content)}

def tester(board: Board):
    game = harness.splice(TEMPLATE, board["logic"])
    return {"report": harness.render(harness.run_tests(game, board["twist"]))}

builder = StateGraph(Board)
builder.add_node("producer", producer)
builder.add_node("coder", coder)
builder.add_node("tester", tester)
builder.add_edge(START, "producer")
builder.add_edge("producer", "coder")
builder.add_edge("coder", "tester")
builder.add_edge("tester", END)
graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "comet-snake"}}
original = graph.invoke({"brief": brief, "twist": "", "logic": "", "report": ""}, config)
history = list(graph.get_state_history(config))
for snapshot in history:
    print("step :", snapshot.metadata["step"], "next :", snapshot.next, "twist :", snapshot.values.get("twist", ""))
before_coder = next(snapshot for snapshot in history if snapshot.next == ("coder",))
original_final = graph.get_state(config).config
print("original :", original["report"])

step : 3 next : () twist : golden
step : 2 next : ('tester',) twist : golden
step : 1 next : ('coder',) twist : golden
step : 0 next : ('producer',) twist : 
step : -1 next : ('__start__',) twist : 
original : 20 of 20 rule tests pass
bot : 40.0 stars on average, best score 152


## b · replay

We invoke the graph with `None` and the config of an older checkpoint. The run continues from that checkpoint's next nodes. Replay before `coder` calls the model again, so the code can change.

In [4]:
count_before = len(calls)
replayed = graph.invoke(None, before_coder.config)
print("replayed :", replayed["report"])
print("new model calls :", len(calls) - count_before)
print("same code :", replayed["logic"] == original["logic"])
print("saved original :", graph.get_state(original_final).values["twist"])

replayed : 20 of 20 rule tests pass
bot : 40.0 stars on average, best score 152
new model calls : 1
same code : False
saved original : golden


## c · forking

We roll back to the checkpoint before the Coder and write another twist, as if the Producer had picked it. The returned config selects a new checkpoint whose next node is `coder`.

In [5]:
forked = graph.update_state(before_coder.config, {"twist": "shield"}, as_node="producer")
count_before = len(calls)
alternative = graph.invoke(None, forked)
print("fork :", alternative["twist"], ":", alternative["report"])
print("new model calls :", len(calls) - count_before)
print("parent unchanged :", graph.get_state(before_coder.config).values["twist"])
print("original build unchanged :", graph.get_state(original_final).values["logic"] == original["logic"])

fork : shield : 21 of 21 rule tests pass
bot : 56.0 stars on average, best score 85
new model calls : 1
parent unchanged : golden
original build unchanged : True


The thread now holds three timelines : the original, the replay and the fork. Each checkpoint names its parent, so the history is a tree.

In [6]:
for snapshot in graph.get_state_history(config):
    parent = (snapshot.parent_config or {"configurable": {"checkpoint_id": "none"}})["configurable"]["checkpoint_id"]
    print(snapshot.config["configurable"]["checkpoint_id"][-6:], "parent", parent[-6:], "step", snapshot.metadata["step"],
          "next", snapshot.next, "twist", snapshot.values.get("twist", ""))

07657b parent e32c62 step 4 next () twist shield
e32c62 parent f0eade step 3 next ('tester',) twist shield
f0eade parent 3d5934 step 2 next ('coder',) twist shield
5e6ae9 parent 1e123f step 4 next () twist golden
1e123f parent 72b0bf step 3 next ('tester',) twist golden
72b0bf parent 3d5934 step 2 next ('coder',) twist golden
3492c3 parent b898d9 step 3 next () twist golden
b898d9 parent 3d5934 step 2 next ('tester',) twist golden
3d5934 parent 9a2896 step 1 next ('coder',) twist golden
9a2896 parent 14114b step 0 next ('producer',) twist 
14114b parent none step -1 next ('__start__',) twist 


We also inspect `DeltaChannel`. Its reducer receives a batch of writes, and the checkpoint saver rebuilds the accumulated list from saved writes. The API is beta on LangGraph 1.2.17.

In [7]:
from typing import Annotated
from langgraph.channels import DeltaChannel

def merge_batches(current, writes):
    return current + [item for batch in writes for item in batch]

class LongThread(TypedDict):
    entries: Annotated[list[str], DeltaChannel(merge_batches, snapshot_frequency=2)]

long_builder = StateGraph(LongThread)
long_builder.add_node("append", lambda state: {"entries": ["tests passed"]})
long_builder.add_edge(START, "append")
long_builder.add_edge("append", END)
long_graph = long_builder.compile(checkpointer=InMemorySaver())
long_config = {"configurable": {"thread_id": "delta-thread"}}
long_graph.invoke({"entries": ["build 1 started"]}, long_config)
long_graph.invoke({"entries": ["build 2 started"]}, long_config)
print("delta restored :", long_graph.get_state(long_config).values["entries"])
print("delta history :", [snapshot.values["entries"] for snapshot in long_graph.get_state_history(long_config) if "entries" in snapshot.values])

delta restored : ['build 1 started', 'tests passed', 'build 2 started', 'tests passed']
delta history : [['build 1 started', 'tests passed', 'build 2 started', 'tests passed'], ['build 1 started', 'tests passed', 'build 2 started'], ['build 1 started', 'tests passed'], ['build 1 started', 'tests passed'], ['build 1 started'], []]


## Things to try

Optional : we can change a brief, a thread id or a returned update and compare the saved state.